Импорт необходимых библиотек

In [1]:
from datetime import datetime, timedelta
from functools import partial
from itertools import product
from typing import Any, Callable, Dict, List, Optional, Tuple

import yfinance
import pandas as pd
import numpy as np
import talib


from backtesting import Backtest, Strategy

from sklearn.model_selection import TimeSeriesSplit, train_test_split


Параметры разделения данных

In [2]:
TEST_SIZE = 100
TRAIN_SIZE = 500

Импорт исторических данных

In [3]:
ticker = 'AAPL'
last_n_days = 1700
time_frame = '1d'

start_date = datetime.now() - timedelta(days=last_n_days)
end_date = datetime.now()

df = yfinance.download(
    tickers=ticker,
    start=start_date,
    end=end_date,
    interval=time_frame
)

df = df.droplevel(1, axis=1)
df = df.drop(columns='Adj Close')
df.columns = df.columns.str.lower()
df = df.sort_index()


[*********************100%***********************]  1 of 1 completed


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1170 entries, 2020-05-14 to 2025-01-07
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   close   1170 non-null   float64
 1   high    1170 non-null   float64
 2   low     1170 non-null   float64
 3   open    1170 non-null   float64
 4   volume  1170 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 54.8 KB


In [5]:
df.head()

Price,close,high,low,open,volume
Date,,,,,
2020-05-14,77.385002,77.447502,75.382500,76.127502,158929200
2020-05-15,76.927498,76.974998,75.052498,75.087502,166348400
2020-05-18,78.739998,79.125000,77.580002,78.292503,135178400
2020-05-19,78.285004,79.629997,78.252502,78.757500,101729600
2020-05-20,79.807503,79.879997,79.129997,79.169998,111504800


Создаем три вида стратегии

In [6]:
RETURN_FORMAT = ['open', 'high', 'low', 'close', 'volume', 'signal']


def buy_and_hold_strategy(df: pd.DataFrame) -> pd.DataFrame:
    """Стратегия 'Купил и держу'"""
    df = df.copy()
    df['signal'] = 1
    
    return df[RETURN_FORMAT]


def bbands_strategy(
        df: pd.DataFrame,
        timeperiod: int,
        nbdevup: float,
        nbdevdn: float
) -> pd.DataFrame:
    """Стратегия на основе полос Боллинджера"""
    df = df.copy()

    # Создаем индикатор по параметрам
    df['u_bound'], _, df['l_bound'] = talib.BBANDS(
        real=df['close'],
        timeperiod=timeperiod,
        nbdevup=nbdevup,
        nbdevdn=nbdevdn,
        matype=talib.MA_Type.SMA
    )
    
    # Инициируем поле 'signal'
    df['signal'] = np.nan
    
    # Сигнал на покупку
    df['signal'] = np.where(df['close'] < df['l_bound'], 1, np.nan)
    
    # Сигнал на продажу
    df['signal'] = np.where(df['close'] > df['u_bound'], -1, df['signal'])
    
    # Заполняем пространство между сигналами значением предыдущего сигнала
    df['signal'] = df['signal'].ffill(limit_area='inside')
    
    # Явно добавлем отсутствие сигнала в начале, когда сигналов нет
    df['signal'] = df['signal'].fillna(0)
    
    return df[RETURN_FORMAT]


def rsi_strategy(
        df: pd.DataFrame,
        timeperiod: int,
        upper_bound: int,
        lower_bound: int
) -> pd.DataFrame:
    df = df.copy()

    # Создаем индикатор по параметрам
    df['rsi'] = talib.RSI(real=df['close'], timeperiod=timeperiod)
    
    # Инициируем поле 'signal'
    df['signal'] = np.nan

    # Сигнал на покупку
    df['signal'] = np.where(df['rsi'] < lower_bound, 1, np.nan)
    
    # Сигнал на продажу
    df['signal'] = np.where(df['rsi'] > upper_bound, -1, df['signal'])

    # Заполняем пространство между сигналами значением предыдущего сигнала
    df['signal'] = df['signal'].ffill(limit_area='inside')

    # Явно добавлем отсутствие сигнала в начале, когда сигналов нет
    df['signal'] = df['signal'].fillna(0)
    
    return df[RETURN_FORMAT]


Создаем универсальный вспомогательный класс TestStrategy для бэктеста

In [7]:
class TestStrategy(Strategy):
    
    def init(self):
        self.signal = self.I(lambda: self.data.Signal)
        self.previous_signal = 0
        self.size = 0.1

    def next(self):
        current_signal = self.signal[-1]

        if current_signal != self.previous_signal:
            if current_signal == 1:
                if self.position.is_short:
                    self.position.close()
                    
                if not self.position.is_long:
                    self.buy(size=self.size)
                    
            elif current_signal == -1:
                if self.position.is_long:
                    self.position.close()
                   
                if not self.position.is_short:
                    self.sell(size=self.size)
                    
            elif current_signal == 0:
                if self.position:
                    self.position.close()

        self.previous_signal = current_signal

Функция бэктеста стратегии

In [8]:
def backtest_strategy(df, strategy_func, params=None):
    params = params or {}
    # Применяем стратегию с переданными параметрами
    signal_df = strategy_func(df.copy(), **params)
    
    # Подготовка данных для бэктеста
    signal_df.columns = signal_df.columns.str.capitalize()

    # Создаем объект класса Backtest с текущей стратегией
    bt = Backtest(signal_df, TestStrategy, cash=500000, commission=.002, exclusive_orders=True, margin=0.1)

    # Запускаем бэктест
    stats = bt.run()
    return stats

Функция обучения (подбор параметров) на исторических данных

In [31]:
def train(
    df: pd.DataFrame,
    strategy_func: Callable[[Any], pd.DataFrame],
    params_lst: Optional[List[Dict[str, Any]]] = None,
    performance_function_nm: str = 'Sortino Ratio'
) -> Callable[[Any], pd.DataFrame]:
    performance_function_nm
    df = df.copy()
    
    # Для хранения лучших параметров и лучшего результата
    best_strategy = None
    best_performance = float('-inf')

    params_lst = params_lst or [{}]

    for params in params_lst:
        stats = backtest_strategy(df, strategy_func, params)
        performance = stats[performance_function_nm]

        # Сравниваем с лучшим результатом и сохраняем лучшие параметры
        if performance > best_performance:
            best_performance = performance
            best_strategy = partial(strategy_func, **params)

    return best_strategy, best_performance

На валидационной выборке проверяем как обучились на трейне

In [23]:
def validation(
    df: pd.DataFrame,
    strategy_func: Callable[[Any], pd.DataFrame],
    strategy_params: List[Dict[str, Any]],
    performance_function_nm: str = 'Sortino Ratio'
) -> List[Tuple[float, float]]:
    df = df.copy()
    performance_metrics = []
    tscv = TimeSeriesSplit(test_size=TEST_SIZE, max_train_size=TRAIN_SIZE)

    for i, (training, validation) in enumerate(tscv.split(df)):
        print(f'batch: {i}')

        train_df = df.iloc[training]
        print(f"train:\n{train_df.index.to_series().agg(['min', 'max', 'count'])}")

        best_strategy, train_performance = train(
            df=train_df,
            strategy_func=strategy_func,
            params_lst=strategy_params,
            performance_function_nm=performance_function_nm
        )
        print(f'train_performance: {train_performance}')
        
        val_df = df.iloc[validation]
        print(f"validation:\n{val_df.index.to_series().agg(['min', 'max', 'count'])}")
        val_performance = backtest_strategy(val_df, best_strategy)[performance_function_nm]
        print(f'val_performance: {val_performance}', end='\n' * 2)

        performance_metrics.append((train_performance, val_performance))

    return performance_metrics

Для каждой стратегии создаем пространство параметров, которое будем тестировать

In [24]:
performance_function_nm = 'Sortino Ratio'

timeperiods = [7, 10, 14, 21, 28]

nbdevups = [1.0, 1.2, 1.5, 1.7, 1.96, 2.1, 2.5, 2.8]
nbdevdns = [1.0, 1.2, 1.5, 1.7, 1.96, 2.1, 2.5, 2.8]

bbands_strategy_params = [
    {
        'timeperiod': tp,
        'nbdevup': up,
        'nbdevdn': dn
    }
    for tp, up, dn
    in product(timeperiods, nbdevups, nbdevdns)
]

lower_bounds = [20, 25, 30, 35, 40]
upper_bounds = [60, 65, 70, 75, 80]

rsi_strategy_params = [
    {
        'timeperiod': tp,
        'lower_bound': lb,
        'upper_bound': ub
    }
    for tp, lb, ub
    in product(timeperiods, lower_bounds, upper_bounds)
]

Планируем какие стратегии и с какими параметрами тестировать

In [25]:
experiment_set = {
    'buy_and_hold_strategy': (buy_and_hold_strategy, None),
    'bbands_strategy': (bbands_strategy, bbands_strategy_params),
    'rsi_strategy': (rsi_strategy, rsi_strategy_params)
}



Делим данные на обучающую и тестовую выборки

In [26]:
train_idx, test_idx = train_test_split(df.index, test_size=TEST_SIZE, shuffle=False)

print('train:', pd.Series(train_idx).agg(['min', 'max', 'count']), sep='\n', end='\n' * 2)
print('test:', pd.Series(test_idx).agg(['min', 'max', 'count']), sep='\n')

train:
min      2020-05-14 00:00:00
max      2024-08-14 00:00:00
count                   1070
Name: Date, dtype: object

test:
min      2024-08-15 00:00:00
max      2025-01-07 00:00:00
count                    100
Name: Date, dtype: object


Проверяем эффективность стратегии на валидационных выборках

In [ ]:
train_df = df.loc[train_idx]
val_results = {}

for strategy_nm, (strategy_func, params) in experiment_set.items():
    val_results[strategy_nm] = validation(train_df, strategy_func, params)

Выбираем лучшую стратегию по результатам валидации

In [28]:
best_result = (None, float('-inf'))

for strategy_nm, strategy_result in val_results.items():
    print(strategy_nm)
    avg_result = pd.DataFrame(strategy_result, columns=['train_performance', 'val_performance']).mean()
    print(avg_result, end='\n' * 2)

    current_result = avg_result['val_performance'].squeeze()

    if best_result[1] < current_result:
        best_result = (strategy_nm, current_result)

buy_and_hold_strategy
train_performance    0.418378
val_performance      3.667521
dtype: float64

bbands_strategy
train_performance    1.616076
val_performance      0.000000
dtype: float64

rsi_strategy
train_performance         inf
val_performance      0.924471
dtype: float64



Финальное тестирование лучшей стратегии на отложенной выборке

In [32]:
print(f'Best strategy: {best_result}', end='\n' * 2)

strategy_func, params = experiment_set[best_result[0]]

best_strategy, train_performance = train(
    df=train_df.tail(TRAIN_SIZE),
    strategy_func=strategy_func,
    params_lst=params,
    performance_function_nm=performance_function_nm
)

test_df = df.loc[test_idx]
test_performance = backtest_strategy(test_df, best_strategy)[performance_function_nm]

print('train_performance:', train_performance, sep=' ')
print('test_performance:', test_performance, sep=' ')


Best strategy: ('buy_and_hold_strategy', np.float64(3.6675207592850407))

train_performance: 0.8081901632404292
test_performance: 1.5259470255769534
